# ETL de Pessoas — VOBYS / Estado de Goiás

**Candidato:** *Gabriel Farias Tranquilli*  
**Projeto:** Implantação do Sistema de Gestão de Pessoas para 40 órgãos do executivo estadual  
**Tecnologias:** Python · Pandas · SQLite · CSV

---

## Contexto

A VOBYS foi contratada pelo Estado de Goiás para implantar um **Sistema de Gestão de Pessoas** atendendo:
- **40 órgãos** do executivo estadual
- ~**166.261 servidores**
- **3 projetos paralelos**: RH/Folha, eSocial, Gestão Estratégica de Pessoas
- Destino final: **banco Oracle** único e orquestrado

Este notebook demonstra a ETL do **Projeto 1 — Migração de Pessoas**, com dados fictícios representativos do cenário real.

## Arquitetura da ETL

```
┌─────────────────┐     ┌──────────────────────┐     ┌─────────────────────┐
│   CSV / Fonte   │────▶│   EXTRACT            │────▶│  Checkpoint JSON    │
│  (dados brutos) │     │  - Leitura em chunks │     │  (retomada após     │
└─────────────────┘     │  - Checkpoint        │     │   falha de rede)    │
                        └──────────┬───────────┘     └─────────────────────┘
                                   │
                                   ▼
                        ┌──────────────────────┐     ┌─────────────────────┐
                        │   TRANSFORM          │────▶│  quarentena.csv     │
                        │  - Normalização      │     │  (registros inváli- │
                        │  - Validação         │     │   dos para análise) │
                        │  - Deduplicação      │     └─────────────────────┘
                        │  - Padronização      │
                        └──────────┬───────────┘
                                   │
                                   ▼
                        ┌──────────────────────┐     ┌─────────────────────┐
                        │   LOAD               │────▶│  SQLite (Oracle-    │
                        │  - Schema 3FN        │     │  compatível)        │
                        │  - UPSERT            │     │  Schema normalizado │
                        │  - Log de execução   │     └─────────────────────┘
                        └──────────────────────┘
```

In [1]:
# ── Configuração inicial ─────────────────────────────────────────
import sys, os
sys.path.insert(0, '../scripts')

import pandas as pd
import sqlite3
import json

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 40)
pd.set_option('display.float_format', '{:.2f}'.format)

DATA_PATH = '../data/servidores_raw.csv'
DB_PATH   = '../output/gestao_pessoas.db'
QUAR_PATH = '../output/quarentena.csv'
REL_PATH  = '../output/relatorio_execucao.csv'

print('Ambiente configurado ✓')

Ambiente configurado ✓


---
## 1. EXTRACT — Geração e Leitura dos Dados Fonte

In [4]:
# Gera dados fictícios com imperfeições intencionais e executa ETL completa

# Limpa os argumentos antes de importar
# Demo:
sys.argv = ['run_etl.py', '--gerar-dados']

# Completo:
sys.argv = ['run_etl.py', '--gerar-dados', '--modo', 'completo']

from run_etl import main

# from extract import reset_checkpoint # Para demonstração do zero
# reset_checkpoint()

metricas = main()

2026-03-31 17:29:27,095 | INFO     | [extract] Checkpoint removido — execução iniciará do zero.
2026-03-31 17:29:27,097 | INFO     | ============================================================
2026-03-31 17:29:27,098 | INFO     |   INICIANDO ETL DE PESSOAS — VOBYS / ESTADO DE GOIÁS
2026-03-31 17:29:27,099 | INFO     |   MODO: COMPLETO
2026-03-31 17:29:27,100 | INFO     | ============================================================
2026-03-31 17:29:27,101 | INFO     | [main] Gerando dados fictícios — modo 'completo'...
[generate_data] Modo COMPLETO — gerando 166,261 registros (Anexo I)...
  ✓ CGE                199 registros gerados
  ✓ PGE                367 registros gerados
  ✓ SEAD              2692 registros gerados
  ✓ ECONOMIA          3108 registros gerados
  ✓ DETRAN            1274 registros gerados
  ✓ SSP               1586 registros gerados
  ✓ CBM               2675 registros gerados
  ✓ DGPP              3316 registros gerados
  ✓ DGPC              4738 registros gerados

In [5]:
# ── Inspecionando os dados brutos ────────────────────────────────
df_raw = pd.read_csv(DATA_PATH)
print(f"Shape: {df_raw.shape}")
print(f"\nColunas: {list(df_raw.columns)}")
df_raw.head(10)

Shape: (166261, 14)

Colunas: ['matricula', 'nome', 'cpf', 'data_nascimento', 'sexo', 'escolaridade', 'cargo', 'orgao', 'vinculo', 'data_admissao', 'salario_bruto', 'email', 'telefone', 'ativo']


,matricula,nome,cpf,data_nascimento,sexo,escolaridade,cargo,orgao,vinculo,data_admissao,salario_bruto,email,telefone,ativo
0,GO0001000,Felipe Nunes,253.037.604-60,25-05-1973,M,Ensino Médio,Coordenador,Cge,Comissionado,2001-12-14,"15906,50",felipe.nunes@cge.go.gov.br,6193316125,1
1,GO0001001,Alexandre Lima,689.789.352-54,19-06-1975,F,Superior Incompleto,Coordenador,CGE,Estatutário,12-10-2021,"10154,60",alexandre.lima@cge.go.gov.br,6198290411,1
2,GO0001000,RODRIGO NASCIMENTO,568.322.810-71,27-09-1983,F,Superior Completo,Professor,CGE,Comissionado,NaN,24317,rodrigo.nascimento@cge.go.gov.br,(64) 93587595,1
3,GO0001003,Ana Ferreira,282.260.775-67,13/11/1975,M,Mestrado,Analista,Cge,Celetista,01/09/2009,"2685,10",ana.ferreira@cge.go.gov.br,NaN,1
4,GO0001004,Henrique Araújo,160.627.124-37,30/06/1977,F,Superior Incompleto,Auditor,CGE,Temporário,03-11-1992,"R$ 24739,87",henrique.araújo@cge.go.gov.br,NaN,1
5,GO0001005,Fernanda Moura,085.420.918-24,1962-08-02,M,Ensino Fundamental,Agente Penitenciário,CGE,Temporário,23-02-2006,1474.41,fernanda.moura@cge.go.gov.br,NaN,1
6,GO0001006,Juliana Ferreira,431.285.120-13,20-11-1997,M,Especialização,Perito,Cge,Temporário,1993-01-19,8369.43,juliana.ferreira@cge.go.gov.br,NaN,0
7,GO0001007,Aline Campos,762.902.718-96,1991-05-28,F,Ensino Médio,Médico,CGE,Estagiário,1997-05-07,"22645,40",aline.campos@cge.go.gov.br,NaN,1
8,GO0001008,Ricardo Souza,539.520.842-50,10/09/1986,F,Superior Incompleto,Analista,Cge,Estatutário,2013-08-02,"16453,10",ricardo.souza@cge.go.gov.br,(62) 95495334,1
9,GO0001009,Camila Santos,982.819.128-08,29-06-1978,F,Doutorado,Auditor,CGE,Temporário,26-10-2006,24321.53,camila.santos@cge.go.gov.br,(62) 99071057,1


In [6]:
# ── Exemplos de imperfeições nos dados fonte ─────────────────────
print("═" * 60)
print("EXEMPLOS DE IMPERFEIÇÕES (propositais) nos dados brutos")
print("═" * 60)

# 1. Formatos de data mistos
print("\n📅 Formatos de data mistos (data_nascimento):")
print(df_raw['data_nascimento'].dropna().unique()[:8])

# 2. Salários em formatos variados
print("\n💰 Salários em formatos variados:")
print(df_raw['salario_bruto'].unique()[:8])

# 3. Órgãos com inconsistência de caixa
print("\n🏛️ Órgãos com inconsistência (caixa/espaços):")
print(df_raw['orgao'].unique()[:12])

# 4. Nulos
print("\n⚠️ Valores nulos por coluna:")
print(df_raw.isnull().sum()[df_raw.isnull().sum() > 0])

════════════════════════════════════════════════════════════
EXEMPLOS DE IMPERFEIÇÕES (propositais) nos dados brutos
════════════════════════════════════════════════════════════

📅 Formatos de data mistos (data_nascimento):
['25-05-1973' '19-06-1975' '27-09-1983' '13/11/1975' '30/06/1977'
 '1962-08-02' '20-11-1997' '1991-05-28']

💰 Salários em formatos variados:
['15906,50' '10154,60' '24317' '2685,10' 'R$ 24739,87' '1474.41' '8369.43'
 '22645,40']

🏛️ Órgãos com inconsistência (caixa/espaços):
['Cge' 'CGE' 'CGE ' ' CGE' 'cge' 'Pge' 'PGE ' 'PGE' ' PGE' 'pge' 'sead'
 'SEAD ']

⚠️ Valores nulos por coluna:
cpf                 4765
data_nascimento     8333
sexo                8293
escolaridade        8286
cargo               8461
vinculo             8166
data_admissao       8231
email               2568
telefone           55684
dtype: int64


In [7]:
# ── Checkpoint: demonstração de robustez ─────────────────────────
import json
checkpoint_path = '../output/checkpoint.json'
with open(checkpoint_path) as f:
    cp = json.load(f)

print("📌 Estado do checkpoint após execução:")
for k, v in cp.items():
    print(f"   {k}: {v}")

print("\n💡 Se a ETL fosse interrompida por erro de rede,")
print("   ela retomaria da linha", cp.get('last_row_extracted', '?'), "automaticamente.")

📌 Estado do checkpoint após execução:
   last_row_extracted: 166261
   timestamp: 2026-03-31T17:29:33.511859
   source: G:\Outros computadores\Meu laptop\AnÃ¡lise e Desenvolvimento de Sistemas\Vagas de emprego\Teste Vobys\etl_vobys\notebook\..\data\servidores_raw.csv
   status: extraction_complete

💡 Se a ETL fosse interrompida por erro de rede,
   ela retomaria da linha 166261 automaticamente.


---
## 2. TRANSFORM — Normalização, Validação e Qualidade de Dados

In [8]:
# ── Comparação: dados brutos × normalizados ──────────────────────
conn = sqlite3.connect(DB_PATH)
df_norm = pd.read_sql("SELECT * FROM servidores LIMIT 10", conn)
conn.close()

print("DADOS APÓS NORMALIZAÇÃO (servidores no banco):")
df_norm[['matricula','nome','cpf','data_nascimento','sexo','salario_bruto','email']].head(10)

DADOS APÓS NORMALIZAÇÃO (servidores no banco):


,matricula,nome,cpf,data_nascimento,sexo,salario_bruto,email
0,GO0002052,Lucas Moreira,321.819.600-13,2000-04-25,None,22378.15,lucas.moreira@sead.go.gov.br
1,GO0001001,Alexandre Lima,816.184.959-31,1975-06-19,F,10154.60,alexandre.lima@cge.go.gov.br
2,GO0001002,Vinicius Costa,030.564.139-53,1966-04-06,F,4939.28,vinicius.costa@secult.go.gov.br
3,GO0001003,Ana Ferreira,018.451.462-70,1975-11-13,M,2685.10,ana.ferreira@cge.go.gov.br
4,GO0001004,Henrique Araújo,822.782.489-63,1977-06-30,F,24739.87,None
5,GO0001005,Fernanda Moura,997.376.311-65,1962-08-02,M,1474.41,fernanda.moura@cge.go.gov.br
6,GO0001006,Juliana Ferreira,602.606.474-68,1997-11-20,M,8369.43,juliana.ferreira@cge.go.gov.br
7,GO0001007,Aline Campos,985.435.346-24,1991-05-28,F,22645.40,aline.campos@cge.go.gov.br
8,GO0001009,Camila Santos,923.226.025-63,1978-06-29,F,24321.53,camila.santos@cge.go.gov.br
9,GO0001011,Maria Monteiro,577.738.721-48,1978-04-20,F,7377.00,maria.monteiro@cge.go.gov.br


In [9]:
# ── Exemplos específicos de normalização ────────────────────────
print("\n📋 NORMALIZAÇÃO APLICADA")
print("-" * 55)
exemplos = [
    ("Nome com caixa mista",  "  carlos SILVA  ",        "Carlos Silva"),
    ("Data dd/mm/yyyy",        "15/03/1985",              "1985-03-15"),
    ("Data yyyy-mm-dd",        "1990-07-22",              "1990-07-22"),
    ("Data dd-mm-yyyy",        "03-11-2001",              "2001-11-03"),
    ("Salário com R$",         "R$ 4.250,75",             "4250.75"),
    ("Salário com vírgula",    "2890,00",                 "2890.0"),
    ("Órgão minúsculo",        "seduc",                   "SEDUC"),
    ("Órgão com espaço",       " ECONOMIA ",              "ECONOMIA"),
    ("Telefone só dígitos",    "62991234567",             "(62) 99123-4567"),
    ("CPF sem formatação",     "12345678901",             "123.456.789-01"),
]
df_ex = pd.DataFrame(exemplos, columns=['Transformação','Valor Bruto','Valor Normalizado'])
print(df_ex.to_string(index=False))


📋 NORMALIZAÇÃO APLICADA
-------------------------------------------------------
       Transformação      Valor Bruto Valor Normalizado
Nome com caixa mista   carlos SILVA        Carlos Silva
     Data dd/mm/yyyy       15/03/1985        1985-03-15
     Data yyyy-mm-dd       1990-07-22        1990-07-22
     Data dd-mm-yyyy       03-11-2001        2001-11-03
      Salário com R$      R$ 4.250,75           4250.75
 Salário com vírgula          2890,00            2890.0
     Órgão minúsculo            seduc             SEDUC
    Órgão com espaço        ECONOMIA           ECONOMIA
 Telefone só dígitos      62991234567   (62) 99123-4567
  CPF sem formatação      12345678901    123.456.789-01


In [10]:
# ── Registros em quarentena ──────────────────────────────────────
df_quar = pd.read_csv(QUAR_PATH)
print(f"Total em quarentena: {len(df_quar)} registros")
print()

# Categorias de erro
import re
categorias = []
for erros in df_quar['erros']:
    if 'cpf_duplicado' in str(erros):    categorias.append('CPF Duplicado')
    elif 'matricula_duplicada' in str(erros): categorias.append('Matrícula Duplicada')
    elif 'cpf_nulo' in str(erros):       categorias.append('CPF Nulo')
    elif 'nome_nulo' in str(erros):      categorias.append('Nome Nulo')
    elif 'orgao_invalido' in str(erros): categorias.append('Órgão Inválido')
    else:                                categorias.append('Outros')

df_quar['categoria'] = categorias
print("Distribuição por tipo de erro:")
print(df_quar['categoria'].value_counts().to_string())
print()
df_quar[['linha_origem','matricula_raw','nome_raw','cpf_raw','categoria']].head(10)

Total em quarentena: 17240 registros

Distribuição por tipo de erro:
categoria
CPF Duplicado          8288
CPF Nulo               4765
Matrícula Duplicada    4187



,linha_origem,matricula_raw,nome_raw,cpf_raw,categoria
0,3,GO0001000,Rodrigo Nascimento,568.322.810-71,Matrícula Duplicada
1,28,GO0001022,Larissa Campos,407.852.519-95,Matrícula Duplicada
2,30,GO0001029,Eduardo Costa,580.093.680-38,CPF Duplicado
3,36,GO0001016,Cristina Nunes,923.630.230-99,CPF Duplicado
4,37,GO0001036,Lucas Souza,NaN,CPF Nulo
5,40,GO0001000,Lucas Almeida,806.447.790-39,Matrícula Duplicada
6,48,GO0001006,Rafael Araújo,NaN,CPF Nulo
7,61,GO0001024,Cristina Moura,074.303.884-25,Matrícula Duplicada
8,67,GO0001066,Thiago Araújo,741.208.773-47,CPF Duplicado
9,68,GO0001067,Alexandre Almeida,NaN,CPF Nulo


---
## 3. LOAD — Schema Normalizado e Integridade Referencial

In [11]:
# ── Schema normalizado (3FN) ─────────────────────────────────────
conn = sqlite3.connect(DB_PATH)

# Tabelas de referência criadas automaticamente
tabelas = ['orgaos','vinculos','cargos','escolaridades','servidores','etl_execucoes']
print("TABELAS NO BANCO (schema 3FN):")
for tab in tabelas:
    count = pd.read_sql(f"SELECT COUNT(*) as n FROM {tab}", conn)['n'][0]
    print(f"  {tab:<20} {count:>6} registros")

conn.close()

TABELAS NO BANCO (schema 3FN):
  orgaos                   39 registros
  vinculos                  5 registros
  cargos                   20 registros
  escolaridades             7 registros
  servidores           164443 registros
  etl_execucoes             7 registros


In [12]:
# ── Tabelas de referência (dimensões) ────────────────────────────
conn = sqlite3.connect(DB_PATH)

print("Órgãos carregados:", end=' ')
df_org = pd.read_sql("SELECT * FROM orgaos ORDER BY sigla", conn)
print(list(df_org['sigla']))

print("\nVínculos:", list(pd.read_sql("SELECT descricao FROM vinculos", conn)['descricao']))
print("Cargos:", list(pd.read_sql("SELECT descricao FROM cargos ORDER BY descricao", conn)['descricao']))

conn.close()

Órgãos carregados: ['ABC', 'AGR', 'AGRODEFESA', 'CASA CIVIL', 'CBM', 'CGE', 'DETRAN', 'DGPC', 'DGPP', 'DPE-GO', 'ECONOMIA', 'EMATER', 'FAPEG', 'GOIAS TURISMO', 'GOIASPREV', 'GOINFRA', 'JUCEG', 'PGE', 'PM', 'RETOMADA', 'SEAD', 'SEAPA', 'SECAMI', 'SECOM', 'SECTI', 'SECULT', 'SEDF', 'SEDS', 'SEDUC', 'SEEL', 'SEINFRA', 'SEMAD', 'SERINT', 'SES', 'SGG', 'SIC', 'SSP', 'UEG', 'VICEGOV']

Vínculos: ['Estatutário', 'Comissionado', 'Celetista', 'Estagiário', 'Temporário']
Cargos: ['Advogado', 'Agente', 'Agente Penitenciário', 'Analista', 'Assistente', 'Auditor', 'Contador', 'Coordenador', 'Delegado', 'Enfermeiro', 'Engenheiro', 'Escrivão', 'Especialista', 'Fiscal', 'Gestor', 'Inspetor', 'Médico', 'Perito', 'Professor', 'Técnico']


In [13]:
# ── Query analítica: servidores por órgão ────────────────────────
conn = sqlite3.connect(DB_PATH)

query = """
    SELECT 
        o.sigla                          AS orgao,
        COUNT(s.id)                      AS total_servidores,
        SUM(CASE WHEN s.ativo=1 THEN 1 ELSE 0 END) AS ativos,
        ROUND(AVG(s.salario_bruto), 2)   AS salario_medio,
        ROUND(MIN(s.salario_bruto), 2)   AS salario_minimo,
        ROUND(MAX(s.salario_bruto), 2)   AS salario_maximo
    FROM servidores s
    JOIN orgaos o ON s.id_orgao = o.id
    GROUP BY o.sigla
    ORDER BY total_servidores DESC
    LIMIT 15
"""

df_analise = pd.read_sql(query, conn)
conn.close()

print("ANÁLISE: Servidores por Órgão")
df_analise

ANÁLISE: Servidores por Órgão


,orgao,total_servidores,ativos,salario_medio,salario_minimo,salario_maximo
0,GOIASPREV,70409,52584,13135.53,1320.00,24999.93
1,SEDUC,44510,33285,13185.34,1320.00,24999.00
2,PM,11492,8598,13101.99,1320.51,24998.08
3,SES,7482,5655,13231.18,1320.94,25000.00
4,DGPC,4669,3521,13256.21,1335.93,24994.56
5,DGPP,3267,2459,13185.36,1332.41,24997.88
6,ECONOMIA,3071,2324,13414.05,1325.48,24991.26
7,SEAD,2655,1971,13167.12,1321.00,24996.62
8,CBM,2647,2002,13214.61,1327.00,24977.13
9,UEG,1980,1519,13440.15,1360.66,24997.07


In [14]:
# ── Query analítica: distribuição por vínculo ─────────────────────
conn = sqlite3.connect(DB_PATH)

df_vinc = pd.read_sql("""
    SELECT 
        COALESCE(v.descricao, 'Não informado') AS vinculo,
        COUNT(*) AS total,
        ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM servidores), 1) AS pct
    FROM servidores s
    LEFT JOIN vinculos v ON s.id_vinculo = v.id
    GROUP BY v.descricao
    ORDER BY total DESC
""", conn)

conn.close()
print("Distribuição por Vínculo:")
df_vinc

Distribuição por Vínculo:


,vinculo,total,pct
0,Comissionado,31445,19.10
1,Celetista,31335,19.10
2,Estatutário,31251,19.00
3,Estagiário,31181,19.00
4,Temporário,31085,18.90
5,Não informado,8146,5.00


---
## 4. Métricas de Execução e Relatório

In [15]:
# ── Relatório consolidado de execução ────────────────────────────
df_rel = pd.read_csv(REL_PATH)
print("RELATÓRIO DE EXECUÇÃO ETL")
df_rel

RELATÓRIO DE EXECUÇÃO ETL


,etapa,metrica,valor
0,geral,execucao_timestamp,2026-03-31 17:31:15
1,geral,tempo_total_segundos,103.564
2,extract,registros_extraidos,166261
3,extract,linhas_ignoradas_checkpoint,0
4,extract,tempo_extract_segundos,1.2284
5,transform,registros_validos,149021
6,transform,registros_quarentena,17240
7,transform,taxa_rejeicao_pct,10.37
8,transform,tempo_transform_segundos,52.3003
9,load,registros_inseridos,15445


In [16]:
# ── Histórico de execuções no banco ──────────────────────────────
conn = sqlite3.connect(DB_PATH)
df_exec = pd.read_sql("SELECT * FROM etl_execucoes ORDER BY id DESC", conn)
conn.close()

print("Log de execuções ETL registrado no banco:")
df_exec

Log de execuções ETL registrado no banco:


,id,iniciado_em,finalizado_em,total_extraido,total_valido,total_carregado,total_erros,status
0,7,2026-03-31 17:30:25,2026-03-31 17:31:15,166261,149021,149020,17241,sucesso
1,6,2026-03-31 17:25:35,2026-03-31 17:25:35,0,0,0,0,sucesso
2,5,2026-03-31 17:18:56,2026-03-31 17:19:31,165761,148617,148617,17144,sucesso
3,4,2026-03-31 15:36:35,2026-03-31 15:36:35,500,445,445,55,sucesso
4,3,2026-03-31 14:57:24,2026-03-31 14:57:24,500,445,445,55,sucesso
5,2,2026-03-31 14:56:59,2026-03-31 14:56:59,0,0,0,0,sucesso
6,1,2026-03-27 15:12:49,2026-03-27 15:12:49,500,445,445,55,sucesso


In [17]:
# ── Resumo visual das métricas ────────────────────────────────────
m_ext = metricas['extract']
m_trn = metricas['transform']
m_ld  = metricas['load']

print("\n" + "═"*55)
print("  RESUMO DE EXECUÇÃO ETL — VOBYS / Estado de Goiás")
print("═"*55)
print(f"  {'EXTRACT':}")
print(f"    Registros lidos:          {m_ext['total_lido']:>6}")
print(f"    Tempo:                    {m_ext['tempo_segundos']:>6.4f}s")
print()
print(f"  {'TRANSFORM':}")
print(f"    Registros válidos:        {m_trn['total_validos']:>6}")
print(f"    Registros em quarentena:  {m_trn['total_quarentena']:>6}")
print(f"    Taxa de rejeição:         {m_trn['taxa_rejeicao_pct']:>5.1f}%")
print(f"    Tempo:                    {m_trn['tempo_segundos']:>6.4f}s")
print()
print(f"  {'LOAD':}")
print(f"    Inseridos:                {m_ld['inseridos']:>6}")
print(f"    Atualizados (UPSERT):     {m_ld['atualizados']:>6}")
print(f"    Tempo:                    {m_ld['tempo_segundos']:>6.4f}s")
print("═"*55)


═══════════════════════════════════════════════════════
  RESUMO DE EXECUÇÃO ETL — VOBYS / Estado de Goiás
═══════════════════════════════════════════════════════
  EXTRACT
    Registros lidos:          166261
    Tempo:                    1.2284s

  TRANSFORM
    Registros válidos:        149021
    Registros em quarentena:   17240
    Taxa de rejeição:          10.4%
    Tempo:                    52.3003s

  LOAD
    Inseridos:                 15445
    Atualizados (UPSERT):     133575
    Tempo:                    50.0353s
═══════════════════════════════════════════════════════


---
## 5. Demonstração de Robustez (Checkpoint)

O mecanismo de **checkpoint** garante que, em caso de interrupção (ex.: queda de rede, timeout de banco), a ETL retome do ponto exato onde parou — sem reprocessar registros já carregados.

In [18]:
# ── Simulação de retomada após falha ─────────────────────────────
import json

# Força um checkpoint parcial simulando falha na linha 250
checkpoint_simulado = {
    'last_row_extracted': 250,
    'timestamp': '2026-01-15T10:30:00',
    'source': '../data/servidores_raw.csv',
    'status': 'error',
    'error': 'ConnectionError: timeout após 30s'
}

with open('../output/checkpoint.json', 'w') as f:
    json.dump(checkpoint_simulado, f, indent=2)

print("🔴 Simulando falha na linha 250...")
print(json.dumps(checkpoint_simulado, indent=2))

🔴 Simulando falha na linha 250...
{
  "last_row_extracted": 250,
  "timestamp": "2026-01-15T10:30:00",
  "source": "../data/servidores_raw.csv",
  "status": "error",
  "error": "ConnectionError: timeout ap\u00f3s 30s"
}


In [19]:
# Retomada: a ETL lê o checkpoint e pula as primeiras 250 linhas
from extract import extract

print("🟢 Retomando ETL a partir do checkpoint...")
registros_retomada, metricas_retomada = extract('../data/servidores_raw.csv', chunk_size=100)

print(f"\n✅ Registros processados na retomada: {metricas_retomada['total_lido']}")
print(f"   Linhas ignoradas (já processadas): {metricas_retomada['linhas_ignoradas_por_checkpoint']}")

🟢 Retomando ETL a partir do checkpoint...
2026-03-31 17:49:28,359 | INFO     | [extract] Checkpoint encontrado: {'last_row_extracted': 250, 'timestamp': '2026-01-15T10:30:00', 'source': '../data/servidores_raw.csv', 'status': 'error', 'error': 'ConnectionError: timeout após 30s'}
2026-03-31 17:49:28,360 | WARNING  | [extract] Retomando extração a partir da linha 250 (checkpoint de 2026-01-15T10:30:00).
2026-03-31 17:49:29,215 | INFO     | [extract] Total de linhas no arquivo: 166261
2026-03-31 17:49:43,893 | INFO     | [extract] Métricas: {'etapa': 'extract', 'total_lido': 166011, 'tempo_segundos': 15.5244, 'linhas_ignoradas_por_checkpoint': 250}

✅ Registros processados na retomada: 166011
   Linhas ignoradas (já processadas): 250


---
## 6. Considerações sobre Escalabilidade para 166.261 Servidores

Para escalar esta ETL ao volume real do projeto:

| Aspecto | Solução atual (PoC) | Recomendação para produção |
|---|---|---|
| **Orquestração** | Script Python | Apache Airflow com DAGs por projeto |
| **Processamento** | Pandas (memória) | PySpark / Pandas em chunks |
| **Banco destino** | SQLite | Oracle (mesmo schema 3FN) |
| **Monitoramento** | Logs + CSV | Grafana + tabela etl_execucoes |
| **Paralelismo** | Sequencial | 3 DAGs paralelas (RH, eSocial, GEP) |
| **Segurança** | Sem criptografia | Colunas CPF/salário com Oracle TDE |
| **Qualidade** | Quarentena CSV | Great Expectations + alertas |

### Riscos e Mitigações

| Risco | Probabilidade | Impacto | Mitigação |
|---|---|---|---|
| Dados de órgãos inconsistentes entre sistemas | Alta | Alto | Tabela de-para com validação prévia |
| CPF inválido/duplicado em grande volume | Alta | Alto | Quarentena + workflow de correção com responsável do órgão |
| Sobreposição de registros entre os 3 projetos | Média | Crítico | Chave de negócio única (CPF) + execução orquestrada |
| Interrupção durante carga de 166k registros | Média | Alto | Checkpoint + UPSERT idempotente |
| LGPD: exposição de dados sensíveis | Baixa | Crítico | Mascaramento em ambientes de teste, acesso por perfil |
